# ParkQ — Beijing Universal Studios (Jupyter Notebook)

This notebook runs the **classical route-optimization workflow** from the supplied ParkQ project, step by step.

It uses the original project modules (`parkq.py` and `data_adapter.py`) and the CSV data included in the original ZIP. It does **not** launch the Streamlit web app.

## Important notes about the original ZIP

The original ZIP does not contain `attractions.csv` or `walking_times_template.csv`, although the original Streamlit `app.py` expects those files. To keep the ZIP's source files untouched, this notebook:

1. Calculates each attraction's mean wait from the six included `line_up_*_all.csv` files, ignoring non-numeric wait entries.
2. Reads ride durations from the included `ride_durations.csv`.
3. Reads the actual supplied pairwise walking matrix from `walking_time-matrix.csv`.
4. Passes those in-memory data structures to the original functions in `parkq.py`.

The averages are based on the bundled sample records. They are **not live wait times** and are not adjusted to the time of day. The route result is a prototype estimate, not an operational guarantee.

The notebook follows the Streamlit app's classical baseline: it builds the QUBO, exhaustively enumerates ride subsets, computes a shortest route, and validates the full time budget. It does **not** execute QAOA or use a quantum computer.

## How to use

- Put this `.ipynb` file in the same folder as `parkq.py`, `data_adapter.py`, and the CSV files from the original ZIP.
- In VS Code, open this notebook and select the Python environment for this project as its kernel.
- Run the cells from top to bottom.
- Edit the settings cell to change the starting attraction, time budget, or attraction preferences.

## 1. Environment setup

For this notebook, the essential third-party package is `pandas`. VS Code's Jupyter extension also needs an IPython kernel. In the VS Code Terminal, with the project's virtual environment activated, run:

```powershell
python -m pip install pandas ipykernel
```

You do not need Streamlit to run this notebook.

In [14]:
from pathlib import Path
import sys
import pandas as pd

# This notebook is designed to sit in the root of the extracted project folder.
PROJECT_DIR = Path.cwd().resolve()
required_files = [
    "parkq.py",
    "data_adapter.py",
    "ride_durations.csv",
    "walking_time-matrix.csv",
]
missing = [name for name in required_files if not (PROJECT_DIR / name).is_file()]

if missing:
    raise FileNotFoundError(
        "The notebook is not running from the ParkQ project folder. "
        f"Missing here: {missing}. Move the notebook into the folder that contains "
        "parkq.py and the CSV files, then reopen it in VS Code."
    )

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f"Project folder: {PROJECT_DIR}")
print("Project source files found.")

Project folder: C:\Users\Xuanhe\Downloads\demo-repository-Final_version\demo-repository-Final_version
Project source files found.


## 2. Check the included data files

This cell checks that all six queue-observation CSV files are present. These are the files used to calculate the mean wait values below.

In [15]:
queue_files = sorted(PROJECT_DIR.glob("line_up_*_all.csv"))
if len(queue_files) != 6:
    raise FileNotFoundError(
        f"Expected the six bundled line_up_*_all.csv files, but found {len(queue_files)}. "
        "Check that the original ZIP was fully extracted."
    )

for path in queue_files:
    print(f"{path.name}: {path.stat().st_size:,} bytes")

print(f"\nQueue files found: {len(queue_files)}")

line_up_10_28_all.csv: 7,894 bytes
line_up_10_29_all.csv: 7,028 bytes
line_up_10_30_all.csv: 12,262 bytes
line_up_10_31_all.csv: 12,194 bytes
line_up_11_01_all.csv: 13,559 bytes
line_up_11_02_all.csv: 12,043 bytes

Queue files found: 6


## 3. Prepare the attraction data in memory

The original ZIP has no `attractions.csv`. This cell recreates the fields the ParkQ optimizer needs without writing a new CSV or changing any original project file. Non-numeric wait entries (for example, `CLOSED`) are excluded from the mean.

In [16]:
# Attraction names are the names paired with R_1 ... R_8 in the project's build_data.py.
RIDE_NAMES = {
    "R_1": "Despicable Me Minion Mayhem",
    "R_2": "Loop-Dee Doop-Dee",
    "R_3": "Harry Potter and the Forbidden Journey",
    "R_4": "Flight of the Hippogriff",
    "R_5": "Jurassic World Adventure",
    "R_6": "Jurassic Flyers",
    "R_7": "Transformer: Battle for the AllSpark",
    "R_8": "Lanterns of Legendary Legends",
}

queue_frames = []
for path in queue_files:
    frame = pd.read_csv(path, encoding="utf-8-sig")
    expected_columns = {"id", "time", "wait"}
    if not expected_columns.issubset(frame.columns):
        raise ValueError(
            f"{path.name} must contain columns {sorted(expected_columns)}; "
            f"found {list(frame.columns)}"
        )
    frame["wait_numeric"] = pd.to_numeric(frame["wait"], errors="coerce")
    frame = frame.dropna(subset=["id", "wait_numeric"])
    frame = frame[frame["wait_numeric"] >= 0]
    queue_frames.append(frame[["id", "wait_numeric"]])

all_wait_samples = pd.concat(queue_frames, ignore_index=True)
wait_summary = (
    all_wait_samples.groupby("id")["wait_numeric"]
    .agg(wait_minutes="mean", wait_sample_count="count")
)

ride_durations = pd.read_csv(PROJECT_DIR / "ride_durations.csv", encoding="utf-8-sig")
if not {"ride_id", "duration_time_min"}.issubset(ride_durations.columns):
    raise ValueError("ride_durations.csv must contain ride_id and duration_time_min")

ride_durations["ride_id"] = ride_durations["ride_id"].astype(str).str.strip()
ride_durations["duration_time_min"] = pd.to_numeric(
    ride_durations["duration_time_min"], errors="raise"
)
ride_durations = ride_durations.set_index("ride_id")

ride_ids = list(ride_durations.index)
missing_wait_ids = sorted(set(ride_ids) - set(wait_summary.index))
missing_names = sorted(set(ride_ids) - set(RIDE_NAMES))
if missing_wait_ids or missing_names:
    raise ValueError(
        f"Missing wait data for {missing_wait_ids}; missing attraction names for {missing_names}"
    )

rides_df = pd.DataFrame({
    "ride_id": ride_ids,
    "ride_name": [RIDE_NAMES[ride_id] for ride_id in ride_ids],
    "wait_minutes": [float(wait_summary.loc[ride_id, "wait_minutes"]) for ride_id in ride_ids],
    "ride_minutes": [float(ride_durations.loc[ride_id, "duration_time_min"]) for ride_id in ride_ids],
    "wait_sample_count": [int(wait_summary.loc[ride_id, "wait_sample_count"]) for ride_id in ride_ids],
})

# parkq.py expects a list of dictionaries with ride_id, wait_minutes and ride_minutes.
rides = rides_df.to_dict(orient="records")

print(f"Numeric wait observations used: {len(all_wait_samples):,}")
print("Attraction data prepared in memory; no CSV files were created or changed.")

Numeric wait observations used: 2,630
Attraction data prepared in memory; no CSV files were created or changed.


In [13]:
display(rides_df.style.format({"wait_minutes": "{:.2f}", "ride_minutes": "{:.1f}"}))

AttributeError: The '.style' accessor requires jinja2

## 4. Load the supplied walking-time matrix

The original `walking_time-matrix.csv` is loaded with the project's `data_adapter.py` function. The ParkQ route validator expects a dictionary keyed by `(from_id, to_id)`, so the nested matrix is converted to that format.

In [ ]:
from data_adapter import build_walking_matrix

walking_nested = build_walking_matrix(PROJECT_DIR / "walking_time-matrix.csv")

if set(walking_nested) != set(ride_ids):
    raise ValueError(
        "The walking matrix ride IDs do not match ride_durations.csv. "
        f"Matrix IDs: {sorted(walking_nested)}; ride IDs: {sorted(ride_ids)}"
    )

walking = {
    (from_id, to_id): float(walking_nested[from_id][to_id])
    for from_id in ride_ids
    for to_id in ride_ids
}

walking_df = pd.DataFrame(
    [[walking[(from_id, to_id)] for to_id in ride_ids] for from_id in ride_ids],
    index=ride_ids,
    columns=ride_ids,
)

print(f"Walking-time pairs loaded: {len(walking)}")
display(walking_df)

Walking-time pairs loaded: 64


,R_1,R_2,R_3,R_4,R_5,R_6,R_7,R_8
R_1,0.0,7.0,5.0,8.0,8.0,5.0,7.0,4.0
R_2,7.0,0.0,10.0,10.0,4.0,2.0,1.0,6.0
R_3,5.0,10.0,0.0,1.0,7.0,9.0,11.0,6.0
R_4,8.0,10.0,1.0,0.0,7.0,9.0,11.0,5.0
R_5,8.0,4.0,7.0,7.0,0.0,2.0,4.0,3.0
R_6,5.0,2.0,9.0,9.0,2.0,0.0,3.0,4.0
R_7,7.0,1.0,11.0,11.0,4.0,3.0,0.0,6.0
R_8,4.0,6.0,6.0,5.0,3.0,4.0,6.0,0.0


## 5. Choose your route settings

Change these values before running the optimization. Attraction preferences use the same 1–5 scale as the Streamlit sliders in `app.py`. The starting location must be one of `R_1` through `R_8`.

In [ ]:
START_ID = "R_1"               # Your current attraction/location
TIME_BUDGET_MINUTES = 240      # Total available time, in minutes
PREFERENCES = {ride_id: 3 for ride_id in ride_ids}  # Each preference: 1 to 5

# Example: make R_3 more important to you
# PREFERENCES["R_3"] = 5

if START_ID not in ride_ids:
    raise ValueError(f"START_ID must be one of {ride_ids}")
if not isinstance(TIME_BUDGET_MINUTES, int) or TIME_BUDGET_MINUTES < 0:
    raise ValueError("TIME_BUDGET_MINUTES must be a nonnegative integer")
if set(PREFERENCES) != set(ride_ids):
    raise ValueError("PREFERENCES must contain one score for every ride ID")
if any(not 1 <= score <= 5 for score in PREFERENCES.values()):
    raise ValueError("Every preference score must be between 1 and 5")

print("Starting location:", START_ID, "—", RIDE_NAMES[START_ID])
print("Time budget:", TIME_BUDGET_MINUTES, "minutes")
print("Preferences:", PREFERENCES)

Starting location: R_1 — Despicable Me Minion Mayhem
Time budget: 240 minutes
Preferences: {'R_1': 3, 'R_2': 3, 'R_3': 3, 'R_4': 3, 'R_5': 3, 'R_6': 3, 'R_7': 3, 'R_8': 3}


## 6. Build the QUBO model

This uses the original `build_qubo()` function. The QUBO's time constraint includes estimated waiting time plus ride duration. Walking time is **not** part of the QUBO; the route validation step accounts for walking afterward.

In [ ]:
from parkq import build_qubo, project_candidates, validate_candidates

qubo_variable_names, qubo_coefficients, qubo_offset = build_qubo(
    rides=rides,
    preferences=PREFERENCES,
    budget=TIME_BUDGET_MINUTES,
)

print("QUBO variables:", len(qubo_variable_names))
print("Upper-triangle QUBO coefficients:", len(qubo_coefficients))
print("QUBO offset:", qubo_offset)
print("Variable names:", qubo_variable_names)
print("\nThe QUBO is constructed for reference; the baseline below uses classical enumeration.")

QUBO variables: 16
Upper-triangle QUBO coefficients: 136
QUBO offset: 24903183.34339865
Variable names: ['R_1', 'R_2', 'R_3', 'R_4', 'R_5', 'R_6', 'R_7', 'R_8', 'slack_0', 'slack_1', 'slack_2', 'slack_3', 'slack_4', 'slack_5', 'slack_6', 'slack_7']

The QUBO is constructed for reference; the baseline below uses classical enumeration.


## 7. Enumerate candidate attraction subsets and validate routes

There are eight attractions, so there are at most \(2^8 = 256\) possible attraction subsets. The original `project_candidates()` enumerates subsets that fit the waiting-plus-ride budget. `validate_candidates()` then finds the shortest open route from the selected starting point and checks the complete time, including walking.

In [ ]:
from itertools import islice

candidate_count = 2 ** len(rides)
candidates = project_candidates(
    rides=rides,
    preferences=PREFERENCES,
    budget=TIME_BUDGET_MINUTES,
    top_k=candidate_count,
)

validated_results = validate_candidates(
    candidates=candidates,
    rides=rides,
    start=START_ID,
    walking=walking,
    budget=TIME_BUDGET_MINUTES,
)

feasible_results = [result for result in validated_results if result["feasible"]]

print(f"Candidate subsets evaluated: {len(candidates)}")
print(f"Feasible routes after walking-time validation: {len(feasible_results)}")

if not feasible_results:
    print("No non-empty route fits this time budget with the current assumptions.")
    best = None
else:
    # project_candidates() ranks by its objective score; the app chooses
    # the first candidate that also passes full route-time validation.
    best = feasible_results[0]
    print("Best feasible candidate selected.")

Candidate subsets evaluated: 102
Feasible routes after walking-time validation: 90
Best feasible candidate selected.


## 8. Recommended route and time breakdown

In [ ]:
if best is not None:
    route_ids = [START_ID] + best["route"]
    route_names = [RIDE_NAMES[ride_id] for ride_id in route_ids]

    print("Recommended route:")
    print("  " + " -> ".join(f"{ride_id} ({name})" for ride_id, name in zip(route_ids, route_names)))
    print()
    print(f"Total estimated time: {best['total_minutes']:.1f} minutes")
    print(f"  Waiting: {best['wait_minutes']:.1f} minutes")
    print(f"  Riding:  {best['play_minutes']:.1f} minutes")
    print(f"  Walking: {best['walk_minutes']:.1f} minutes")
    print(f"Time budget: {TIME_BUDGET_MINUTES} minutes")
else:
    print("No route summary to display because no candidate is feasible.")

Recommended route:
  R_1 (Despicable Me Minion Mayhem) -> R_8 (Lanterns of Legendary Legends) -> R_7 (Transformer: Battle for the AllSpark) -> R_2 (Loop-Dee Doop-Dee)

Total estimated time: 227.6 minutes
  Waiting: 194.6 minutes
  Riding:  22.0 minutes
  Walking: 11.0 minutes
Time budget: 240 minutes


## 9. Build the visit timeline

In [ ]:
if best is not None:
    by_id = {ride["ride_id"]: ride for ride in rides}
    elapsed = 0.0
    current_location = START_ID
    timeline_rows = []

    # If the starting attraction is selected, its wait and ride time are counted
    # before walking to the next attraction, matching the original app.py logic.
    if START_ID in best["selected"]:
        ride = by_id[START_ID]
        wait = ride["wait_minutes"]
        play = ride["ride_minutes"]
        timeline_rows.append({
            "ride_id": START_ID,
            "attraction": ride["ride_name"],
            "walk_to_attraction_min": 0.0,
            "arrival_after_start_min": 0.0,
            "wait_min": round(wait, 1),
            "ride_min": round(play, 1),
            "finish_after_start_min": round(wait + play, 1),
        })
        elapsed += wait + play

    for ride_id in best["route"]:
        travel = walking[(current_location, ride_id)]
        elapsed += travel
        arrival = elapsed
        wait = by_id[ride_id]["wait_minutes"]
        play = by_id[ride_id]["ride_minutes"]
        finish = elapsed + wait + play

        timeline_rows.append({
            "ride_id": ride_id,
            "attraction": by_id[ride_id]["ride_name"],
            "walk_to_attraction_min": round(travel, 1),
            "arrival_after_start_min": round(arrival, 1),
            "wait_min": round(wait, 1),
            "ride_min": round(play, 1),
            "finish_after_start_min": round(finish, 1),
        })
        elapsed = finish
        current_location = ride_id

    timeline_df = pd.DataFrame(timeline_rows)
    display(timeline_df)
else:
    print("No timeline available.")

,ride_id,attraction,walk_to_attraction_min,arrival_after_start_min,wait_min,ride_min,finish_after_start_min
0,R_1,Despicable Me Minion Mayhem,0.0,0.0,57.8,10.0,67.8
1,R_8,Lanterns of Legendary Legends,4.0,71.8,30.6,3.0,105.3
2,R_7,Transformer: Battle for the AllSpark,6.0,111.3,58.0,6.0,175.3
3,R_2,Loop-Dee Doop-Dee,1.0,176.3,48.3,3.0,227.6


## 10. Compare the best feasible candidate combinations

The table below shows up to 20 feasible combinations, ordered by the original candidate ranking.

In [ ]:
if feasible_results:
    comparison_columns = [
        "selected", "route", "score", "dwell_minutes",
        "wait_minutes", "play_minutes", "walk_minutes", "total_minutes",
    ]
    comparison_df = pd.DataFrame(feasible_results)
    display(comparison_df[comparison_columns].head(20))
else:
    print("No feasible candidate combinations to compare.")

,selected,route,score,dwell_minutes,wait_minutes,play_minutes,walk_minutes,total_minutes
0,"[R_1, R_2, R_7, R_8]","[R_8, R_7, R_2]",220.541,218,194.59,22.0,11.0,227.59
1,"[R_1, R_2, R_5, R_8]","[R_8, R_5, R_2]",220.106,225,198.94,24.0,11.0,233.94
2,"[R_2, R_5, R_7, R_8]","[R_8, R_5, R_7, R_2]",220.081,221,199.19,20.0,12.0,231.19
3,"[R_1, R_2, R_4, R_8]","[R_2, R_8, R_4]",219.983,221,200.17,19.0,18.0,237.17
4,"[R_2, R_4, R_7, R_8]","[R_7, R_2, R_8, R_4]",219.958,217,200.42,15.0,19.0,234.42
5,"[R_1, R_2, R_8]","[R_8, R_2]",166.341,154,136.59,16.0,10.0,162.59
6,"[R_2, R_7, R_8]","[R_8, R_7, R_2]",166.316,150,136.84,12.0,11.0,159.84
7,"[R_2, R_5, R_8]","[R_8, R_5, R_2]",165.881,157,141.19,14.0,11.0,166.19
8,"[R_2, R_4, R_8]","[R_2, R_8, R_4]",165.758,153,142.42,9.0,18.0,169.42
9,"[R_1, R_7, R_8]","[R_8, R_7]",165.368,166,146.32,19.0,10.0,175.32


## What this notebook does — and does not do

- **Does:** use the bundled queue CSVs, ride durations and walking matrix; call the original ParkQ QUBO and route-validation functions; enumerate attraction subsets; show the best feasible route and its time breakdown.
- **Does not:** connect to live park wait times, estimate current queue conditions, launch the Streamlit UI, or execute QAOA.
- **Data caveat:** the included queue observations are historical sample records. The notebook averages all numeric observations for each attraction, so those averages do not represent a live or time-of-day-specific wait estimate.
- **Original files:** the notebook does not create or overwrite `attractions.csv` or `walking_times_template.csv`.